### SVG generation & evaluation workflow

In [23]:
### Init...
### df load
### model call vllm
### get df response
### close vllm server
### clean df response
### call siglip
### get score
### close siglip server 
### write output
### new_metric_score (manual)

In [24]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
import torch
import gc
import time
sys.path.append('./utils')

### Define

In [25]:
model_path="./lora/Llama-3.2-3B-Instruct_r256_s2000_i1000_v1"
global model_path

In [26]:
from vllm import LLM, SamplingParams
from vllm_model_server_start import vllm_model_server_start
from start_siglip_server import start_siglip_server
from siglip_class import SVGMetricEvaluator
from terminate_server import terminate_server

### Start vLLM model server

In [27]:
vllm_process= vllm_model_server_start(model_path)

vLLM server started with PID: 20398
Waiting for vLLM model to load...
INFO 04-15 01:00:08 [__init__.py:239] Automatically detected platform cuda.
INFO 04-15 01:00:09 [api_server.py:981] vLLM API server version 0.8.2
INFO 04-15 01:00:09 [api_server.py:982] args: Namespace(subparser='serve', model_tag='./lora/Llama-3.2-3B-Instruct_r256_s2000_i1000_v1', config='', host='127.0.0.1', port=8000, uvicorn_log_level='info', disable_uvicorn_access_log=False, allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key='my-api-key', lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, enable_ssl_refresh=False, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, tool_call_parser=None, tool_parser_plu

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  1.26it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.96it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.81it/s]



INFO 04-15 01:00:21 [loader.py:447] Loading weights took 1.13 seconds
INFO 04-15 01:00:22 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 1.279089 seconds
INFO 04-15 01:00:27 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/9c66c74840/rank_0_0 for vLLM's torch.compile
INFO 04-15 01:00:27 [backends.py:425] Dynamo bytecode transform time: 5.85 s
INFO 04-15 01:00:28 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-15 01:00:32 [monitor.py:33] torch.compile takes 5.85 s in total
INFO 04-15 01:00:33 [kv_cache_utils.py:566] GPU KV cache size: 25,552 tokens
INFO 04-15 01:00:33 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 24.95x
INFO 04-15 01:00:47 [gpu_model_runner.py:1534] Graph capturing finished in 14 secs, took 0.44 GiB
INFO 04-15 01:00:47 [core.py:151] init engine (profile, create kv cache, warmup model) took 25.22 seconds
WARNING 04-15 01:00:47 [config.py:1028] Default samp

INFO:     Started server process [20398]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


INFO 04-15 01:00:57 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO:     127.0.0.1:49144 - "GET /health HTTP/1.1" 200 OK
vLLM server is ready.


### OpenAI style client

In [28]:
from openai import OpenAI
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "my-api-key"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

def get_reponse(description):
    alpaca_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

            ### Instruction:
            Generate a SVG code for the given input:
            
            ### Input:
            {description}
                            
            ### Response:
            """
    
    formatted_input = alpaca_prompt.format(description)
    chat_response = client.chat.completions.create(
        model=model_path,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{formatted_input}"},
            ],
        seed=123
    )
    return chat_response.choices[0].message.content


### Load df

In [29]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)


(75, 7)


,description,gpt_svg,gpt_score_sl,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


### Concurrent calls to vLLM server 

In [30]:
import time
from tqdm import tqdm
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# Wrap tqdm over futures
def parallel_apply_with_tqdm(func, data, max_workers=8):
    results = [None] * len(data)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(func, data[i]): i for i in range(len(data))}
        for future in tqdm(as_completed(futures), total=len(data)):
            idx = futures[future]
            try:
                results[idx] = future.result()
            except Exception as e:
                results[idx] = None
                print(f"Error at index {idx}: {e}")
    return results

# Example usage
start_time = time.time()
df['response_3'] = parallel_apply_with_tqdm(get_reponse, df['description'].tolist(), max_workers=8)

end_time = time.time()
print(f"Total time taken: {end_time - start_time:.2f} seconds")


  0%|                                                    | 0/75 [00:00<?, ?it/s]

INFO 04-15 01:01:06 [chat_utils.py:379] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO 04-15 01:01:06 [logger.py:39] Received request chatcmpl-f41408f7b4d04cd89c0993dbaea4325d: prompt: "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 15 Apr 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBelow is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n            ### Instruction:\n            Generate a SVG code for the given input:\n            \n            ### Input:\n             'Morning dew on grass',\n                            \n            ### Response:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_pe

  1%|▌                                           | 1/75 [00:01<02:22,  1.93s/it]

INFO:     127.0.0.1:49218 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 04-15 01:01:08 [logger.py:39] Received request chatcmpl-3c19deba683d47318834ac0d3e05f051: prompt: "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 15 Apr 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBelow is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n            ### Instruction:\n            Generate a SVG code for the given input:\n            \n            ### Input:\n             'Abstract geometric shapes',\n                            \n            ### Response:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.6, top_p=0.9, top_k=-1, min_p=0.0, seed=123, stop=[], stop_

  4%|█▊                                          | 3/75 [00:02<00:48,  1.49it/s]

...

INFO:     127.0.0.1:49152 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 04-15 01:02:07 [loggers.py:80] Avg prompt throughput: 10.1 tokens/s, Avg generation throughput: 253.5 tokens/s, Running: 2 reqs, Waiting: 0 reqs, GPU KV cache usage: 7.5%, Prefix cache hit rate: 83.1%


 99%|██████████████████████████████████████████▍| 74/75 [01:01<00:02,  2.29s/it]

INFO:     127.0.0.1:49224 - "POST /v1/chat/completions HTTP/1.1" 200 OK


100%|███████████████████████████████████████████| 75/75 [01:02<00:00,  1.20it/s]

INFO:     127.0.0.1:49196 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Total time taken: 62.48 seconds


### Terminate vLLM Server

In [31]:
terminate_server(vllm_process)

INFO 04-15 01:02:08 [launcher.py:74] Shutting down FastAPI HTTP server.


[rank0]:[W415 01:02:08.695688707 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


vLLM running: False, GPU used: 793.18 MB, Below threshold: True


{'vllm_running': False,
 'gpu_memory_used_mb': 793.182208,
 'below_threshold': True}

### Extract csv from response & clean csv

In [32]:
from clean_svg_response import Model
from tqdm import tqdm
tqdm.pandas()
model=Model()
df['svg_3'] = df.progress_apply(lambda row: model.clean_svg(row['response_3']), axis=1)

ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 1, column 2292 (<string>, line 1). Returning default SVG.
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 1, column 2489 (<string>, line 1). Returning default SVG.
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 1, column 2344 (<string>, line 1). Returning default SVG.
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 1, column 2379 (<string>, line 1). Returning default SVG.
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 1, column 2351 (<string>, line 1). Returning default SVG.
ERROR:root:SVG Parse Error: Specification mandates value for attribute ry, line 1, column 2209 (<string>, line 1). Returning default SVG.
ERROR:root:SVG Parse Error: Specification mandates value for attribute cx, line 1, column 2203 (<string>, line 1). Returning default SVG.
ERROR:root:SVG Parse Error: S

### Start sl server

In [33]:
server_process=start_siglip_server()

server starting...
Server running: True


### Get sl score from sl server (sometime getting slow, need to check why...)

In [34]:
import pandas as pd
import httpx
from tqdm import tqdm

API_URL = "http://127.0.0.1:8000/evaluate_svg"
API_KEY = "my-api-key"

def send_request(client, prompt, svg):
    try:
        response = client.post(
            API_URL,
            headers={"x-api-key": API_KEY},
            json={"prompt": prompt, "svg": svg},
            timeout=10.0
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

def evaluate_all(df):
    with httpx.Client() as client:
        results = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating SVGs"):
            result = send_request(client, row["description"], row["svg_3"])
            results.append(result)
        return results

# Run evaluation
results = evaluate_all(df)

# Append score or error
df["score_3"] = [r.get("score") if "score" in r else r.get("error") for r in results]


Evaluating SVGs: 100%|██████████████████████████| 75/75 [00:04<00:00, 15.95it/s]


### Get sl score from sl fn

In [35]:
# from tqdm import tqdm
# tqdm.pandas()
# evaluator=SVGMetricEvaluator()
# df['score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['csv_3']), axis=1)

### Terminate sl server

In [36]:
terminate_server(server_process)

vLLM running: False, GPU used: 739.97 MB, Below threshold: True


{'vllm_running': False,
 'gpu_memory_used_mb': 739.966976,
 'below_threshold': True}

In [37]:
df = df[~df['score_3'].apply(lambda x: isinstance(x, str))]
print(df.shape)
print(df['score_3'].mean())

(75, 10)
0.13550917190789474
